# CSI Training on Kaggle (GPU)
Notebook này tương tự train_on_colab.ipynb nhưng đã được chỉnh để chạy an toàn trên Kaggle Kernel.

## 1) GPU accelerator
Trên Kaggle: Kernel → Settings → Accelerator → GPU. Bật GPU trước khi chạy các cell.

In [ ]:
import importlib.util
import subprocess
import sys
import os

def ensure_torch_from_requirements():
    # Nếu PyTorch không có sẵn, cố gắng cài từ requirements.txt (cẩn thận trên Kaggle).
    if importlib.util.find_spec('torch') is None:
        req = os.path.join(os.getcwd(), 'requirements.txt')
        if os.path.exists(req):
            print('PyTorch không tìm thấy: cài từ requirements.txt (có thể lâu)...')
            subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-r', req])
        else:
            print('PyTorch không tìm thấy và requirements.txt không có. Cài torch cơ bản...')
            subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'torch', 'torchvision', 'torchaudio'])
try:
    ensure_torch_from_requirements()
except Exception as e:
    print('Bước cài đặt thất bại:', e)

import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('PyTorch:', getattr(torch, '__version__', 'n/a'))
print('Using device:', device)
if device.type == 'cuda':
    try:
        print('GPU:', torch.cuda.get_device_name(0))
    except Exception:
        pass
else:
    print('WARNING: GPU không khả dụng — bật GPU trong Kernel Settings nếu cần hiệu năng cao.')

## 2) Đưa project vào Kernel (Kaggle)
Trên Kaggle, thêm repository/dữ liệu như một Dataset (Add Data). Notebook sẽ cố gắng tìm thư mục trong `/kaggle/input` và sao chép các thư mục cần thiết vào `/kaggle/working` để có thể ghi file.

In [ ]:
import os, shutil, sys

def detect_kaggle_dataset():
    root = '/kaggle/input'
    if os.path.exists(root):
        items = [d for d in os.listdir(root) if os.path.isdir(os.path.join(root, d))]
        if not items:
            return None
        if len(items) == 1:
            return items[0]
        for name in items:
            if 'csi' in name.lower() or 'pbl' in name.lower() or 'train' in name.lower():
                return name
        return items[0]
    return None

kaggle_dataset = os.getenv('KAGGLE_DATASET') or detect_kaggle_dataset()
kaggle_input_dir = os.path.join('/kaggle/input', kaggle_dataset) if kaggle_dataset else None

PROJECT_DIR = os.getenv('PROJECT_DIR', os.getcwd())
if kaggle_input_dir and os.path.exists(kaggle_input_dir):
    print('Found dataset at', kaggle_input_dir, '— copying project files to /kaggle/working')
    for name in ('src', 'configs', 'data', 'models', 'requirements.txt'):
        s = os.path.join(kaggle_input_dir, name)
        t = os.path.join('/kaggle/working', name)
        if os.path.exists(s):
            if os.path.exists(t):
                if os.path.isdir(t):
                    shutil.rmtree(t)
                else:
                    os.remove(t)
            if os.path.isdir(s):
                shutil.copytree(s, t)
            else:
                os.makedirs(os.path.dirname(t), exist_ok=True)
                shutil.copy2(s, t)
    PROJECT_DIR = '/kaggle/working'
elif os.path.isdir('src'):
    PROJECT_DIR = os.getcwd()
else:
    print('Không tìm thấy project trong current dir hoặc /kaggle/input. Hãy thêm repo/dữ liệu làm Dataset hoặc đặt KAGGLE_DATASET env var.')

os.chdir(PROJECT_DIR)
sys.path.insert(0, PROJECT_DIR)
print('Project dir:', PROJECT_DIR)

## 3) Cài dependencies & kiểm tra dữ liệu
Cài `requirements.txt` (nếu cần) và kiểm tra các file dữ liệu/config cần thiết.

In [ ]:
import os, subprocess, sys

req_file = os.path.join(PROJECT_DIR, 'requirements.txt')
if os.path.exists(req_file):
    print('Installing requirements from', req_file)
    try:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-r', req_file])
    except Exception as e:
        print('pip install failed:', e)
else:
    print('No requirements.txt found in project dir.')

def find_file(rel_path):
    # Try project dir first, then search in /kaggle/input/*
    p = os.path.join(PROJECT_DIR, rel_path)
    if os.path.exists(p):
        return p
    root = '/kaggle/input'
    if os.path.exists(root):
        for d in os.listdir(root):
            candidate = os.path.join(root, d, rel_path)
            if os.path.exists(candidate):
                return candidate
    return None

required_files = [
    'data/raw/sit.csv',
    'data/raw/stand.csv',
    'configs/train_default.json',
]
missing = [p for p in required_files if find_file(p) is None]
print('Missing files:', missing)
assert not missing, f'Missing required files: {missing}'
print('Data and config OK, ready to train.')

## 4) Train CNN2D (khuyến nghị chạy cell này trước)

In [ ]:
import subprocess, sys, os
cmd = [
    sys.executable, '-m', 'src.train',
    '--config', 'configs/train_default.json',
    '--model-type', 'cnn2d',
    '--epochs', '20',
    '--batch-size', '16',
    '--run-name', 'kaggle_cnn2d',
    '--output-dir', '/kaggle/working/experiments',
]
print('Running:', ' '.join(cmd))
subprocess.check_call(cmd)

## 5) Train LSTM-CNN

In [ ]:
import subprocess, sys, os
cmd = [
    sys.executable, '-m', 'src.train',
    '--config', 'configs/train_default.json',
    '--model-type', 'lstmcnn',
    '--epochs', '30',
    '--batch-size', '16',
    '--run-name', 'kaggle_lstmcnn',
    '--output-dir', '/kaggle/working/experiments',
]
print('Running:', ' '.join(cmd))
subprocess.check_call(cmd)

## 6) Kiểm tra checkpoints và logs

In [ ]:
import os
print('Checkpoints (models/checkpoints):')
cp = '/kaggle/working/models/checkpoints'
if os.path.exists(cp):
    for f in os.listdir(cp):
        print('-', f)
else:
    print(cp, 'not found')

print('
Experiment results (experiments/results):')
er = '/kaggle/working/experiments'
if os.path.exists(er):
    for root, dirs, files in os.walk(er):
        for name in files:
            print('-', os.path.join(root, name))
else:
    print(er, 'not found')

## 7) Lưu outputs
Trên Kaggle, mọi file ghi vào `/kaggle/working` sẽ xuất sang tab 
 để tải về. Nếu muốn, tạo Dataset riêng để lưu model lớn và tái sử dụng.
Không dùng `drive.mount()` trên Kaggle — dùng tính năng Output / Dataset để xuất artifacts.